In [1]:
!pip install -q langchain==0.2.16
!pip install -q langchain-community==0.2.16
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q transformers

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import pipeline

print("Libraries Imported Successfully!")

Libraries Imported Successfully!


In [6]:
from google.colab import files

uploaded = files.upload()

Saving wireless sensor networks notes.pdf to wireless sensor networks notes.pdf


In [7]:
pdf_name = list(uploaded.keys())[0]

loader = PyPDFLoader(pdf_name)

documents = loader.load()

print("Number of Pages:", len(documents))

Number of Pages: 31


In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

print("Total Chunks Created:", len(docs))

Total Chunks Created: 160


In [9]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded Successfully!")

/tmp/ipykernel_4209/1455882494.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embedding = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded Successfully!


In [10]:
vector_db = FAISS.from_documents(
    docs,
    embedding
)

print("FAISS Vector Database Created Successfully!")

FAISS Vector Database Created Successfully!


In [11]:
question = input("Enter your question: ")

results = vector_db.similarity_search(question, k=2)

print("\nMost Relevant Chunks:\n")

for i, doc in enumerate(results, start=1):
    print(f"Chunk {i}:\n")
    print(doc.page_content)
    print("\n" + "-"*80 + "\n")

Enter your question: what is sensor?

Most Relevant Chunks:

Chunk 1:

1.6.1 Sensors 
Sensors are responsible for capturing data from the environment. Diﬀerent types of sensors can 
measure various physical phenomena such as: 
 Temperature Sensors: Measure ambient temperature. 
 Humidity Sensors: Measure the moisture content in the air. 
 Pressure Sensors: Measure atmospheric pressure or pressure changes in ﬂuids. 
 Light Sensors: Measure light intensity, oŌen used in surveillance or environmental 
monitoring. 
1.6.2 Processor

--------------------------------------------------------------------------------

Chunk 2:

o Example: A building's HVAC system can use temperature sensors to adjust heaƟng 
and cooling based on occupancy levels, reducing energy consumpƟon. 
3. Industrial AutomaƟon: 
o Overview: WSNs are deployed in factories and industrial environments to monitor 
machinery, detect faults, and opƟmize producƟon processes. 
o Example: Sensors aƩached to machines can monitor 

In [15]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

context = ""

for doc in results:
    context += doc.page_content + "\n"

prompt = f"""
Context:
{context}

Question:
{question}

Answer:
"""

answer = generator(
    prompt,
    max_new_tokens=100,
    do_sample=False
)

print("\nGenerated Answer:\n")
print(answer[0]["generated_text"])

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Generated Answer:


Context:
1.6.1 Sensors 
Sensors are responsible for capturing data from the environment. Diﬀerent types of sensors can 
measure various physical phenomena such as: 
 Temperature Sensors: Measure ambient temperature. 
 Humidity Sensors: Measure the moisture content in the air. 
 Pressure Sensors: Measure atmospheric pressure or pressure changes in ﬂuids. 
 Light Sensors: Measure light intensity, oŌen used in surveillance or environmental 
monitoring. 
1.6.2 Processor
o Example: A building's HVAC system can use temperature sensors to adjust heaƟng 
and cooling based on occupancy levels, reducing energy consumpƟon. 
3. Industrial AutomaƟon: 
o Overview: WSNs are deployed in factories and industrial environments to monitor 
machinery, detect faults, and opƟmize producƟon processes. 
o Example: Sensors aƩached to machines can monitor temperature and vibraƟon, 
alerƟng maintenance staﬀ when a machine requires servicing before it breaks down.


Question:
what is senso

In [16]:
question = "What is this document about?"

In [23]:
question = "What is this document about?"

results = vector_db.similarity_search(question, k=3)

for doc in results:
    print(doc.page_content)
    print("-" * 80)

 Flash Memory: Used for long-term storage of collected data or system logs. 
1.6.4 Transceiver 
The transceiver handles wireless communicaƟon between the sensor node and other nodes in the 
network. It is responsible for sending and receiving data. Transceivers need to be energy-eﬃcient, as 
communicaƟon is the most energy-intensive operaƟon in WSNs. 
1.6.5 Power Source 
Typically, sensor nodes are powered by baƩeries, but alternaƟve sources like energy harvesƟng
--------------------------------------------------------------------------------
 Dynamic Loading: ConƟki allows dynamic loading and updaƟng of programs over the 
network, which is useful for reconﬁguring or upgrading deployed WSNs without physically 
accessing the nodes. 
5.2.3 Hardware Plaƞorms 
Common hardware plaƞorms for WSNs include: 
 MicaZ: A widely-used plaƞorm that includes an Atmel microcontroller (Atmega128L) and a 
low-power radio module (CC2420). It supports communicaƟon using the IEEE 802.15.4 
protocol.
----

In [24]:
question = "Summarize the document."

In [25]:
question = "Summarize the document."

results = vector_db.similarity_search(question, k=3)

for doc in results:
    print(doc.page_content)
    print("-" * 80)

4. Received Signal Strength Indicator (RSSI): 
o Principle: EsƟmates the distance between nodes based on the strength of the 
received signal, assuming that signal strength diminishes with distance.
--------------------------------------------------------------------------------
4o 
You said: 
make a notes on module 4 in detailed and comprehensive way 
ChatGPT said: 
ChatGPT 
Module 4: LocalizaƟon and Tracking (8 Hours) 
 
This module covers the essenƟal concepts of LocalizaƟon and Tracking in Wireless Sensor Networks 
(WSNs). LocalizaƟon refers to determining the geographic posiƟon of sensor nodes, while tracking 
focuses on monitoring the movement and posiƟon of objects in the network. Both concepts are
--------------------------------------------------------------------------------
 Trace-Based Analysis: Ns2/Ns3 provides detailed trace ﬁles that can be used to analyze 
network performance, including metrics like throughput, delay, and energy consumpƟon. 
5.3.2 Omnet++ 
 Overview: 